# Cost-Sensitive Fraud Risk Model: main pipeline notebook
This ONE notebook grows step by step: each step adds a new section (§1, §2, ...).
**Kaggle:** attach the competition data "IEEE-CIS Fraud Detection", then Run All.
**Laptop:** works on `data/sample/sample_50k.parquet` (made in §1 on Kaggle) or on the full CSVs in `data/raw/`.
Every line has a comment: `#` starts a comment, and Python ignores everything after it on that line.

## §0 Setup: where am I running, and where is the data?

In [ ]:
from pathlib import Path          # Python built-in: Path = an object for file/folder paths (works on Windows + Linux)
import json                       # Python built-in: read/write .json files (text files holding dicts)
import numpy as np                # NumPy library, nickname "np": fast arrays + maths (np.log1p, np.nan)
import pandas as pd               # pandas library, nickname "pd": tables called DataFrames
import matplotlib.pyplot as plt   # matplotlib's plotting part, nickname "plt": draws charts

ON_KAGGLE = Path("/kaggle/input").exists()   # .exists() → True/False; this folder exists only inside Kaggle

if ON_KAGGLE:                                                   # if True → we are on Kaggle
    csv_path = next(Path("/kaggle/input").rglob("train_transaction.csv"))  # rglob = search all subfolders; next() = take the first match
    DATA_DIR = csv_path.parent                                  # .parent = the folder that contains that file
    OUT_DIR = Path("/kaggle/working")                           # Kaggle lets you save files only in this folder
else:                                                           # otherwise → we are on the laptop
    PROJECT_DIR = Path.cwd().parent                             # cwd = current folder (notebooks/); .parent = one level up = project folder
    DATA_DIR = PROJECT_DIR / "data" / "raw"                     # "/" between Paths joins them → project/data/raw
    OUT_DIR = PROJECT_DIR                                       # outputs go into the project folder itself

FIGURES_DIR = OUT_DIR / "reports" / "figures"                   # where plots get saved
FIGURES_DIR.mkdir(parents=True, exist_ok=True)                  # mkdir = make folder; parents=True: also make missing parent folders; exist_ok=True: no error if it exists
(OUT_DIR / "data" / "sample").mkdir(parents=True, exist_ok=True)  # same for the sample-data folder
RANDOM_SEED = 42                                                # fixed seed → the same "random" results every run (reproducible)
print("On Kaggle:", ON_KAGGLE, "| data folder:", DATA_DIR)      # print = show text; commas put spaces between items

In [ ]:
def save_metrics(step_name, numbers):                            # def = define a function; step_name and numbers are its inputs
    """Add this step's numbers to reports/metrics.json (your REAL results file)."""  # docstring = description of the function
    file_name = "metrics_DEMO.json" if DEMO_MODE else "metrics.json"  # demo numbers are fake → keep them in a separate file
    metrics_path = OUT_DIR / "reports" / file_name                # path of the results file
    if metrics_path.exists():                                     # file already there from an earlier step?
        all_metrics = json.loads(metrics_path.read_text())        # read_text = file → string; json.loads = string → dict
    else:                                                         # first time
        all_metrics = {}                                          # {} = empty dict
    all_metrics[step_name] = numbers                              # dict[key] = value → add/replace this step's numbers
    metrics_path.write_text(json.dumps(all_metrics, indent=2))    # json.dumps = dict → text (indent=2 makes it readable); write_text saves it
    print("saved →", metrics_path)                                # confirm where it was saved

## §1 Load, merge, EDA: what does the data look like, and why does accuracy lie?

In [ ]:
def shrink_floats(df):                                           # input: a DataFrame; output: the same table using less RAM
    """float64 (8 bytes per number) → float32 (4 bytes): halves RAM; precision is still plenty."""
    float_columns = df.select_dtypes("float64").columns           # select_dtypes = keep only columns of that type; .columns = their names
    df[float_columns] = df[float_columns].astype("float32")       # .astype = convert type; write the converted columns back
    return df                                                     # return = give the result back to whoever called the function

def make_demo_data(n_rows=40_000):                                # FAKE data with the same column names as IEEE-CIS (practice only)
    rng = np.random.default_rng(RANDOM_SEED)                      # NumPy random generator (seeded → same fake data every run)
    demo = pd.DataFrame({
        "TransactionID": np.arange(2_987_000, 2_987_000 + n_rows),  # ids like the real data
        "TransactionDT": np.sort(rng.integers(86_400, 86_400 * 183, n_rows)),  # seconds over ~6 months, sorted in time
        "TransactionAmt": np.round(rng.lognormal(4, 1, n_rows), 2).astype("float32"),  # skewed amounts (many small, few big)
        "ProductCD": rng.choice(list("WCRHS"), n_rows, p=[0.74, 0.12, 0.06, 0.05, 0.03]),  # product code
        "card1": rng.integers(1000, 18000, n_rows),               # card id-like number
        "card4": rng.choice(["visa", "mastercard", "discover", None], n_rows, p=[0.6, 0.3, 0.05, 0.05]),  # card brand (some missing)
        "P_emaildomain": rng.choice(["gmail.com", "yahoo.com", "hotmail.com", None], n_rows),  # email domain
    })
    for v in range(1, 6):                                         # 5 numeric "V" columns, 70% missing like the real ones
        values = rng.normal(size=n_rows).astype("float32")
        values[rng.random(n_rows) < 0.7] = np.nan
        demo[f"V{v}"] = values
    demo["id_01"] = np.where(rng.random(n_rows) < 0.25, rng.normal(size=n_rows), np.nan).astype("float32")  # identity: ~25% present
    demo["DeviceType"] = np.where(demo["id_01"].notna(), rng.choice(["mobile", "desktop"], n_rows), None)
    risk = (0.9 * (demo["ProductCD"] == "C") + 0.6 * np.log1p(demo["TransactionAmt"]) / 5   # a made-up pattern so models can learn something
            + 0.5 * demo["id_01"].notna() + rng.normal(0, 0.6, n_rows))
    demo["isFraud"] = (risk > np.quantile(risk, 0.965)).astype(int)  # top 3.5% "risk" → fraud = 1
    return demo

DEMO_MODE = False                                                 # becomes True only if no real data is found
sample_file = OUT_DIR / "data" / "sample" / "sample_50k.parquet"  # .parquet = compact, fast table file format
if (DATA_DIR / "train_transaction.csv").exists():                 # full data available? (always on Kaggle)
    transactions_df = shrink_floats(pd.read_csv(DATA_DIR / "train_transaction.csv"))  # pd.read_csv = CSV file → DataFrame
    identity_df = shrink_floats(pd.read_csv(DATA_DIR / "train_identity.csv"))        # second table: device/browser info
    print("transactions:", transactions_df.shape, "| identity:", identity_df.shape)  # .shape = (rows, columns)
    fraud_df = transactions_df.merge(                             # .merge = join two tables (like SQL JOIN)
        identity_df,                                              # the right-hand table
        on="TransactionID",                                       # on = the shared key column used to match rows
        how="left")                                               # how="left" = keep EVERY transaction; NaN where no identity row
    del transactions_df, identity_df                              # del = delete these names → frees RAM (we only need fraud_df now)
elif sample_file.exists():                                        # laptop: the 50K sample downloaded from Kaggle
    fraud_df = pd.read_parquet(sample_file)                       # pd.read_parquet = .parquet file → DataFrame
else:                                                             # laptop with NO data at all → practice mode
    DEMO_MODE = True                                              # from here on, numbers are FAKE (saved to metrics_DEMO.json)
    print("⚠ No data found → DEMO MODE: making a small FAKE fraud-like table so you can run and study every cell.")
    print("  For REAL numbers: run this notebook on Kaggle, or put the Kaggle sample in data/sample/.")
    fraud_df = make_demo_data()                                   # function defined just above
n_merged_columns = fraud_df.shape[1]                              # shape[1] = number of columns; remember it before we add new ones
print("merged table:", fraud_df.shape)                            # expected on full data ≈ (590540, 434)

In [ ]:
fraud_rate = fraud_df["isFraud"].mean()                           # df["col"] = one column; .mean() of 0/1 values = share of 1s = fraud rate
print(fraud_df["isFraud"].value_counts())                         # .value_counts() = how many rows have each value (0s and 1s)
print(f"fraud rate = {fraud_rate:.2%}")                           # f-string: {x:.2%} prints 0.035 as 3.50%

always_legit_accuracy = 1 - fraud_rate                            # a "model" that always says 0 (legit) is right on every legit row
print(f"'never fraud' model → accuracy = {always_legit_accuracy:.2%}, frauds caught = 0")  # high accuracy, zero value

In [ ]:
has_identity_share = fraud_df["id_01"].notna().mean()             # .notna() = True where NOT missing; mean = share of rows with identity info
missing_share = fraud_df.isna().mean()                            # .isna() on the whole table → True where missing; mean per column = share missing
missing_share = missing_share.sort_values(ascending=False)        # sort from most-missing to least-missing
n_mostly_empty = int((missing_share > 0.5).sum())                 # (x > 0.5) → True/False per column; .sum() counts the Trues; int() = plain integer
print(f"rows with identity info: {has_identity_share:.1%}")       # .1% = percent with 1 decimal
print("columns more than 50% empty:", n_mostly_empty, "of", n_merged_columns)
missing_share.head(8)                                             # .head(8) = first 8 rows; the last line of a cell gets displayed

In [ ]:
fraud_by_product = (fraud_df                                      # brackets ( ) let one statement run over several lines
    .groupby("ProductCD")["isFraud"]                              # split rows by product code, look at isFraud inside each group
    .agg(["mean", "count"])                                       # .agg = several summaries at once: mean (= fraud rate) and count (= rows)
    .sort_values("mean", ascending=False))                        # highest fraud rate first
fraud_by_product                                                  # display the small table

In [ ]:
figure, axis = plt.subplots(figsize=(7, 4))                       # plt.subplots = new empty chart; figsize = (width, height) in inches
for label_value, label_name in [(0, "legit"), (1, "fraud")]:      # loop twice: first (0, "legit"), then (1, "fraud")
    is_this_class = fraud_df["isFraud"] == label_value            # == gives True/False per row (a "mask")
    amounts = fraud_df.loc[is_this_class, "TransactionAmt"]       # .loc[rows, column] = pick the rows where mask is True, amount column
    axis.hist(np.log1p(amounts),                                  # hist = histogram; np.log1p(x) = log(1 + x), squashes huge amounts
              bins=60,                                            # 60 bars
              alpha=0.5,                                          # 50% see-through, so both colours are visible
              density=True,                                       # scale each class to area 1, so fraud (rare) and legit are comparable
              label=label_name)                                   # name used in the legend
axis.set_xlabel("log(1 + TransactionAmt)")                        # x-axis title
axis.set_ylabel("density (each class scaled to area 1)")          # y-axis title
axis.legend()                                                     # show the colour key
figure.savefig(FIGURES_DIR / "01_amount_by_class.png", dpi=120, bbox_inches="tight")  # save the picture; dpi = sharpness

In [ ]:
SECONDS_PER_DAY = 24 * 60 * 60                                    # 86,400 seconds in a day
fraud_df["day"] = fraud_df["TransactionDT"] // SECONDS_PER_DAY    # TransactionDT is in seconds; // = whole-number division → day number
daily_fraud_rate = fraud_df.groupby("day")["isFraud"].mean()      # fraud rate for each day
n_days = int(fraud_df["day"].max() - fraud_df["day"].min()) + 1   # .max()/.min() = largest/smallest day → how many days of data
print("data covers", n_days, "days")                              # expected ≈ 182 days (about 6 months)
weekly_average = daily_fraud_rate.rolling(7).mean()               # .rolling(7).mean() = average of each 7-day window (smooths noise)
axis = weekly_average.plot(figsize=(7, 3), title="fraud rate, 7-day average")  # pandas .plot() draws a line chart directly
axis.figure.savefig(FIGURES_DIR / "01_fraud_rate_over_time.png", dpi=120, bbox_inches="tight")

In [ ]:
if ON_KAGGLE:                                                     # only on Kaggle: make the small file for the laptop
    laptop_sample = fraud_df.sample(n=50_000, random_state=RANDOM_SEED)  # .sample(n=...) = pick random rows; 50_000 is just 50000 (underscore for readability)
    laptop_sample.to_parquet(sample_file)                         # save it; download it from Kaggle's Output panel into data/sample/

save_metrics("step01_eda", {                                      # save this step's REAL numbers
    "n_rows": int(len(fraud_df)),                                 # len(table) = number of rows
    "n_columns_after_merge": int(n_merged_columns),
    "fraud_rate": round(float(fraud_rate), 5),                    # round(x, 5) = keep 5 decimals; float() = plain Python number
    "always_legit_accuracy": round(float(always_legit_accuracy), 5),
    "share_rows_with_identity": round(float(has_identity_share), 4),
    "n_columns_over_50pct_missing": n_mostly_empty,
    "n_days": n_days,
})

## §2 Time-based split: train on the past, check on the future
train = first 70% of time · valid = next 15% (tuning, early stopping, calibration) · test = last 15% (touched ONCE, at the end)

In [ ]:
NOT_FEATURES = ["TransactionID",                                  # just a row ID: no meaning, the model would memorise it
                "isFraud",                                        # the answer (label): must never be an input!
                "TransactionDT",                                  # raw time: keeps growing, so future values never appear in training
                "day"]                                            # also time (we made it in §1)
feature_columns = [column for column in fraud_df.columns          # list comprehension: go through every column name...
                   if column not in NOT_FEATURES]                 # ...and keep it only if it is not in the NOT_FEATURES list
text_columns = [column for column in feature_columns              # same pattern: keep columns...
                if not pd.api.types.is_numeric_dtype(fraud_df[column])]  # ...whose type is NOT a number (e.g. "visa", "gmail.com")
for column in text_columns:                                       # loop over each text column
    fraud_df[column] = fraud_df[column].astype("category")        # "category" type = text stored as codes; LightGBM/XGBoost/CatBoost can use it directly
print(len(feature_columns), "features,", len(text_columns), "of them text/category")  # len(list) = number of items

In [ ]:
time_values = fraud_df["TransactionDT"]                           # the time column (seconds)
valid_start_time = time_values.quantile(0.70)                     # .quantile(0.70) = the time before which 70% of transactions happened
test_start_time = time_values.quantile(0.85)                      # the time before which 85% happened

is_train = time_values < valid_start_time                         # True for the oldest 70% of rows
is_valid = (time_values >= valid_start_time) & (time_values < test_start_time)  # & = AND (both conditions True) → the middle 15%
is_test = time_values >= test_start_time                          # True for the newest 15%

train_df = fraud_df[is_train]                                     # df[mask] = keep only the rows where the mask is True
valid_df = fraud_df[is_valid]
test_df = fraud_df[is_test]

for split_name, split_df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:  # loop over 3 (name, table) pairs
    first_day, last_day = int(split_df["day"].min()), int(split_df["day"].max())  # first and last day number in this split
    print(f"{split_name:5s} rows={len(split_df):>7,} days {first_day}-{last_day} "  # :5s = pad name to 5 chars; :>7, = right-align with commas
          f"fraud rate={split_df['isFraud'].mean():.2%}")       # fraud rate inside this split

In [ ]:
X_train, y_train = train_df[feature_columns], train_df["isFraud"]  # X = inputs (features table), y = answers (0/1 labels)
X_valid, y_valid = valid_df[feature_columns], valid_df["isFraud"]  # a, b = c, d → assigns c to a and d to b in one line
X_test, y_test = test_df[feature_columns], test_df["isFraud"]
print("X_train", X_train.shape, "| X_valid", X_valid.shape, "| X_test", X_test.shape)

### Leakage demo (optional, ~5 min on Kaggle): random split vs time split
Same model, same amount of data. The ONLY difference is how we split.
AUC = how well the model ranks fraud above legit (1.0 = perfect, 0.5 = coin toss). Full meaning in Step 7.

In [ ]:
RUN_LEAKAGE_DEMO = True                                           # set to False to skip this slow cell

if RUN_LEAKAGE_DEMO:
    import lightgbm as lgb                                        # LightGBM library (Step 6 explains it); nickname lgb
    from sklearn.model_selection import train_test_split          # scikit-learn helper: RANDOM split of rows
    from sklearn.metrics import roc_auc_score                     # scikit-learn: computes ROC-AUC from labels + scores

    def quick_model_auc(X_fit, y_fit, X_check, y_check):          # train on (X_fit, y_fit), measure AUC on (X_check, y_check)
        quick_model = lgb.LGBMClassifier(n_estimators=300,        # 300 small trees
                                         learning_rate=0.05,      # how big each correction step is
                                         num_leaves=64,           # how complex each tree may be
                                         random_state=RANDOM_SEED,
                                         verbose=-1)              # -1 = don't print training logs
        quick_model.fit(X_fit, y_fit)                             # .fit = learn from data
        fraud_scores = quick_model.predict_proba(X_check)[:, 1]   # predict_proba → [P(legit), P(fraud)] per row; [:, 1] = all rows, column 1 = P(fraud)
        return roc_auc_score(y_check, fraud_scores)               # compare true labels with scores → one number

    time_split_auc = quick_model_auc(X_train, y_train, X_valid, y_valid)  # honest: learn on the past, check on the future

    past_rows = pd.concat([train_df, valid_df])                   # pd.concat = stack tables on top of each other (same 85% of rows)
    random_fit_df, random_check_df = train_test_split(            # shuffle those rows randomly and cut them into two parts
        past_rows,
        test_size=len(valid_df),                                  # check part has the SAME size as valid_df → a fair comparison
        random_state=RANDOM_SEED,
        stratify=past_rows["isFraud"])                            # stratify = keep the same fraud % in both parts
    random_split_auc = quick_model_auc(random_fit_df[feature_columns], random_fit_df["isFraud"],
                                       random_check_df[feature_columns], random_check_df["isFraud"])  # leaky: future rows mixed into training

    print(f"time split AUC   = {time_split_auc:.4f}  (honest)")
    print(f"random split AUC = {random_split_auc:.4f}  (usually HIGHER = too optimistic)")

In [ ]:
split_numbers = {                                                 # this step's REAL numbers
    "train_rows": int(len(train_df)), "valid_rows": int(len(valid_df)), "test_rows": int(len(test_df)),
    "train_fraud_rate": round(float(y_train.mean()), 5),
    "valid_fraud_rate": round(float(y_valid.mean()), 5),
    "test_fraud_rate": round(float(y_test.mean()), 5),
    "n_features": len(feature_columns),
}
if RUN_LEAKAGE_DEMO:                                              # add the demo results only if the demo ran
    split_numbers["leakage_demo_time_split_auc"] = round(float(time_split_auc), 4)
    split_numbers["leakage_demo_random_split_auc"] = round(float(random_split_auc), 4)
save_metrics("step02_split", split_numbers)

## §3 Three boosters on the time split: LightGBM vs XGBoost vs CatBoost (+ class imbalance)
All three: same X_train / X_valid from §2 · early stopping on VALID PR-AUC · test set still untouched.

In [ ]:
import time                                                       # Python built-in: measure how long training takes
import joblib                                                     # save / load trained models as files
import lightgbm as lgb                                            # LightGBM library, nickname lgb
import xgboost as xgb                                             # XGBoost library, nickname xgb
from catboost import CatBoostClassifier                           # CatBoost: import just the classifier class
from sklearn.metrics import average_precision_score, roc_auc_score  # sklearn: PR-AUC (average precision) and ROC-AUC

FAST_MODE = True                                                  # True = quicker first run (bigger steps, fewer rounds); False = slower, usually a bit better
LEARNING_RATE = 0.1 if FAST_MODE else 0.03                        # one-line if/else: step size of each tree
MAX_ROUNDS = 1000 if FAST_MODE else 5000                          # upper limit on trees; early stopping usually stops sooner
PATIENCE = 100 if FAST_MODE else 200                              # stop if valid PR-AUC hasn't improved for this many rounds

n_fraud = int(y_train.sum())                                      # y is 0/1, so the sum = number of fraud rows
n_legit = int(len(y_train) - n_fraud)                             # everything else is legit
imbalance_ratio = n_legit / n_fraud                               # ≈ 27: legit rows per fraud row
USE_CLASS_WEIGHT = False                                          # False = keep natural ratio (honest probabilities); True = weight fraud rows ×ratio
positive_weight = imbalance_ratio if USE_CLASS_WEIGHT else 1.0    # passed to every model as scale_pos_weight
(OUT_DIR / "models").mkdir(parents=True, exist_ok=True)           # folder for saved models
print(f"fraud={n_fraud:,} legit={n_legit:,} ratio={imbalance_ratio:.1f} | weight used={positive_weight:.1f} | FAST_MODE={FAST_MODE}")

In [ ]:
start = time.time()                                               # time.time() = seconds since 1970; difference = duration
lgb_model = lgb.LGBMClassifier(                                   # MAKE: an empty LightGBM model with its settings
    n_estimators=MAX_ROUNDS,                                      # max number of trees
    learning_rate=LEARNING_RATE,                                  # shrinkage per tree
    num_leaves=256,                                               # leaf-wise growth → cap the number of leaves per tree
    min_child_samples=100,                                        # each leaf needs ≥100 rows (stops tiny, memorised leaves)
    subsample=0.8, subsample_freq=1,                              # each tree sees a random 80% of rows (bagging inside boosting)
    colsample_bytree=0.5,                                         # each tree sees a random 50% of columns (de-correlates trees)
    scale_pos_weight=positive_weight,                             # class-imbalance weight (1.0 = off)
    metric="average_precision",                                   # the score to watch = PR-AUC
    random_state=RANDOM_SEED, verbose=-1)
lgb_model.fit(X_train, y_train,                                   # FIT: learn from train...
              eval_set=[(X_valid, y_valid)],                      # ...while scoring on valid after every tree
              callbacks=[lgb.early_stopping(PATIENCE),            # stop when valid PR-AUC stops improving
                         lgb.log_evaluation(100)])                # print progress every 100 trees
lgb_seconds = time.time() - start
print(f"LightGBM best round = {lgb_model.best_iteration_}, time = {lgb_seconds:.0f}s")

In [ ]:
start = time.time()
xgb_model = xgb.XGBClassifier(                                    # MAKE: XGBoost model
    n_estimators=MAX_ROUNDS,
    learning_rate=LEARNING_RATE,
    max_depth=8,                                                  # level-wise growth → cap the depth instead of leaves
    min_child_weight=5,                                           # minimum hessian sum in a leaf (XGBoost's "min rows")
    subsample=0.8, colsample_bytree=0.5,
    tree_method="hist",                                           # histogram binning = fast
    enable_categorical=True,                                      # use our "category" columns directly
    scale_pos_weight=positive_weight,
    eval_metric="aucpr",                                          # XGBoost's name for PR-AUC
    early_stopping_rounds=PATIENCE,                               # in XGBoost ≥2.0 this goes in the constructor
    random_state=RANDOM_SEED)
xgb_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=100)  # FIT with valid monitoring
xgb_seconds = time.time() - start
print(f"XGBoost best round = {xgb_model.best_iteration}, time = {xgb_seconds:.0f}s")

In [ ]:
def to_catboost_format(features_df):                              # CatBoost wants text categories as plain strings, no NaN
    catboost_df = features_df.copy()                              # .copy() so we don't change X_train itself
    for column in text_columns:                                   # text_columns was made in §2
        catboost_df[column] = (catboost_df[column]
                               .astype("object")                  # category → plain Python objects (so we can add a new value)
                               .fillna("missing"))                # NaN → the text "missing" = its own category
    return catboost_df

start = time.time()
cat_model = CatBoostClassifier(                                   # MAKE: CatBoost model
    iterations=MAX_ROUNDS,
    learning_rate=LEARNING_RATE,
    depth=8,                                                      # symmetric trees of depth 8
    eval_metric="PRAUC",                                          # CatBoost's name for PR-AUC
    early_stopping_rounds=PATIENCE,
    scale_pos_weight=positive_weight,
    cat_features=text_columns,                                    # tell CatBoost which columns are categories
    random_seed=RANDOM_SEED,
    verbose=100)
cat_model.fit(to_catboost_format(X_train), y_train,
              eval_set=(to_catboost_format(X_valid), y_valid),
              use_best_model=True)                                # keep the trees up to the best valid round
cat_seconds = time.time() - start
print(f"CatBoost best round = {cat_model.get_best_iteration()}, time = {cat_seconds:.0f}s")

In [ ]:
valid_scores = {                                                  # dict: model name → P(fraud) for every valid row
    "LightGBM": lgb_model.predict_proba(X_valid)[:, 1],          # [:, 1] = column 1 = P(fraud)
    "XGBoost": xgb_model.predict_proba(X_valid)[:, 1],
    "CatBoost": cat_model.predict_proba(to_catboost_format(X_valid))[:, 1],
}
train_seconds = {"LightGBM": lgb_seconds, "XGBoost": xgb_seconds, "CatBoost": cat_seconds}

comparison_rows = []                                              # one dict per model → becomes a table
for model_name, fraud_scores in valid_scores.items():             # .items() = (key, value) pairs of the dict
    comparison_rows.append({
        "model": model_name,
        "valid_pr_auc": round(float(average_precision_score(y_valid, fraud_scores)), 4),  # PR-AUC: main metric
        "valid_roc_auc": round(float(roc_auc_score(y_valid, fraud_scores)), 4),           # ROC-AUC: for comparison
        "mean_predicted_p": round(float(fraud_scores.mean()), 4),  # vs real fraud rate → hint about calibration (Step 9)
        "train_seconds": round(train_seconds[model_name]),
    })
comparison_df = pd.DataFrame(comparison_rows).sort_values("valid_pr_auc", ascending=False)  # list of dicts → table, best first
print(f"real valid fraud rate = {y_valid.mean():.4f}")
comparison_df

In [ ]:
joblib.dump(lgb_model, OUT_DIR / "models" / "lightgbm.joblib")    # joblib.dump(object, path) = save to disk
joblib.dump(xgb_model, OUT_DIR / "models" / "xgboost.joblib")
cat_model.save_model(str(OUT_DIR / "models" / "catboost.cbm"))     # CatBoost has its own save format
save_metrics("step03_models", {
    "fast_mode": FAST_MODE,
    "use_class_weight": USE_CLASS_WEIGHT,
    "imbalance_ratio": round(imbalance_ratio, 2),
    "models": comparison_df.to_dict(orient="records"),           # table → list of dicts (JSON-friendly)
    "best_rounds": {"LightGBM": int(lgb_model.best_iteration_),
                    "XGBoost": int(xgb_model.best_iteration),
                    "CatBoost": int(cat_model.get_best_iteration())},
})

## §4 Metrics: confusion matrix, precision / recall / F1, ROC curve vs PR curve (on VALID)
Test set still untouched. Uses valid_scores (dict of P(fraud) per model) from §3.

In [ ]:
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score,  # brackets ( ) let one import span lines
                             precision_recall_curve, roc_curve)

best_model_name = comparison_df.iloc[0]["model"]                  # .iloc[0] = first row of the table (sorted best PR-AUC first)
best_scores = valid_scores[best_model_name]                       # its P(fraud) for every valid row
predicted_at_half = (best_scores >= 0.5).astype(int)              # the "default" 0.5 threshold (Step 10 replaces it with a cost-based one)

tn, fp, fn, tp = confusion_matrix(y_valid, predicted_at_half).ravel()  # sklearn: [[TN, FP], [FN, TP]] → .ravel() flattens to 4 numbers
print(f"{best_model_name} @ threshold 0.5 → TP={tp:,} FP={fp:,} FN={fn:,} TN={tn:,}")
print(f"precision={precision_score(y_valid, predicted_at_half):.3f}  "
      f"recall={recall_score(y_valid, predicted_at_half):.3f}  "
      f"F1={f1_score(y_valid, predicted_at_half):.3f}  "
      f"FPR={fp / (fp + tn):.4f}")                                  # FPR = false alarms ÷ all real legit

In [ ]:
figure, (roc_axis, pr_axis) = plt.subplots(1, 2, figsize=(12, 4.5))  # 1 row, 2 charts side by side
for model_name, fraud_scores in valid_scores.items():                # one line per model on each chart
    fpr_values, tpr_values, _ = roc_curve(y_valid, fraud_scores)      # ROC: for every threshold → (FPR, TPR=recall)
    roc_axis.plot(fpr_values, tpr_values, label=f"{model_name} (ROC-AUC {roc_auc_score(y_valid, fraud_scores):.3f})")
    precision_values, recall_values, _ = precision_recall_curve(y_valid, fraud_scores)  # PR: for every threshold → (precision, recall)
    pr_axis.plot(recall_values, precision_values, label=f"{model_name} (PR-AUC {average_precision_score(y_valid, fraud_scores):.3f})")

roc_axis.plot([0, 1], [0, 1], linestyle="--", color="grey", label="coin toss (0.5)")  # the ROC baseline
roc_axis.set_xlabel("FPR (share of legit flagged)"); roc_axis.set_ylabel("TPR = recall"); roc_axis.legend()
pr_axis.axhline(y_valid.mean(), linestyle="--", color="grey", label=f"baseline = fraud rate {y_valid.mean():.3f}")  # PR baseline
pr_axis.set_xlabel("recall"); pr_axis.set_ylabel("precision"); pr_axis.legend()
figure.savefig(FIGURES_DIR / "04_roc_vs_pr_curves.png", dpi=120, bbox_inches="tight")

In [ ]:
def recall_at_precision(labels, scores, min_precision=0.5):       # business question: "if 50% of flags must be real, how much fraud do we catch?"
    precision_values, recall_values, _ = precision_recall_curve(labels, scores)
    ok = precision_values >= min_precision                        # True where precision is high enough
    return float(recall_values[ok].max()) if ok.any() else 0.0    # best recall among those points

metric_rows = []
for model_name, fraud_scores in valid_scores.items():
    metric_rows.append({
        "model": model_name,
        "pr_auc": round(float(average_precision_score(y_valid, fraud_scores)), 4),
        "roc_auc": round(float(roc_auc_score(y_valid, fraud_scores)), 4),
        "recall_at_precision_50": round(recall_at_precision(y_valid, fraud_scores, 0.5), 4),
    })
metrics_df = pd.DataFrame(metric_rows).sort_values("pr_auc", ascending=False)
save_metrics("step04_metrics", {
    "best_model": best_model_name,
    "at_threshold_0_5": {"tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
                         "precision": round(float(precision_score(y_valid, predicted_at_half)), 4),
                         "recall": round(float(recall_score(y_valid, predicted_at_half)), 4)},
    "per_model": metrics_df.to_dict(orient="records"),
})
metrics_df